# Smart Collision Avoidance - Bulletproof 90-Degree Rotational Escape Live Demo

This notebook implements a **Bulletproof Rotational Escape Algorithm**:
1. **3-Frame Debounce Confirmation**: Confirms `BLOCKED` only after 3 consecutive frames to eliminate camera noise.
2. **Cumulative Rotational Reverse (`REVERSE_TURNING` for 1.8s)**: Reverses with MAX steering (`steering = -1.0`), turning the car's nose 90 degrees away from the wall/obstacle (prevents left-right wiggling in place!).
3. **Hardware Pause (`PAUSE` for 0.3s)**: Stops throttle to allow smooth motor & steering transitions.
4. **Straight Forward Recovery (`CHECK_FORWARD` for 1.2s)**: Drives straight forward in the new direction (`steering = 0.0`). If still blocked in a tight corner, turns another 90 degrees until a clear path is found!


### 1. Setup Environment & Pre-load CUDA Libraries


In [ ]:
import os
import sys
import cv2
import time
import ctypes
import numpy as np
import threading
import base64
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

# Pre-load CUDA libraries into RTLD_GLOBAL symbol table so TensorRT C++ shared libraries load smoothly
cuda_lib64 = "/usr/local/cuda/lib64"
for lib_name in ["libcudart.so", "libcudnn.so", "libcuda.so", "libnvinfer.so"]:
    lib_path = os.path.join(cuda_lib64, lib_name)
    if os.path.exists(lib_path):
        try:
            ctypes.CDLL(lib_path, mode=ctypes.RTLD_GLOBAL)
        except Exception:
            pass

# Add parent search paths to sys.path
pass # curr = Path.cwd()
for p in [curr, curr.parent, curr.parent.parent]:
    if p.exists() and str(p) not in sys.path:
        pass

import rospy
import onnxruntime as ort
from sensor_msgs.msg import Image as ROSImage
from jetracer_ai.hardware import NvidiaRacecar

try:
    from jetracer_ai.utils import bgr8_to_jpeg, preprocess_onnx
except ImportError:
    from utils import bgr8_to_jpeg, preprocess_onnx

# Initialize ROS Node
try:
    rospy.init_node('collision_avoidance_camera_evade_notebook', anonymous=True, disable_signals=True)
    print("[+] ROS Node initialized successfully!")
except Exception as e:
    print(f"[*] ROS Node notice: {e}")


### 2. Configure & Load TensorRT InferenceSession (`Collision Avoidance MobileNet Model`)


In [ ]:
# 1. ALWAYS load .onnx model path into ONNX Runtime
onnx_path = os.path.join(Path.cwd(), "models/urban_traffic/best_model_mobilenet.onnx")
if not os.path.exists(onnx_path):
    onnx_path = os.path.join(Path.cwd().parent, "models/urban_traffic/best_model_mobilenet.onnx")
if not os.path.exists(onnx_path):
    onnx_path = os.path.join(Path.cwd(), "best_model_resnet18.onnx")

if not os.path.exists(onnx_path):
    raise FileNotFoundError(f"[!] ONNX model not found: {onnx_path}. Please run train_model_onnx.ipynb first!")

print(f"[*] Loading Pure ONNX Model: {onnx_path}")

# 2. Configure TensorRT Execution Provider (ORT will automatically compile & cache the .engine file on GPU)
trt_options = {
    'device_id': 0,
    'trt_max_workspace_size': 1073741824, # 1GB
    'trt_fp16_enable': True,
    'trt_engine_cache_enable': True,
    'trt_engine_cache_path': str(Path.cwd()),
}

available_providers = ort.get_available_providers()
print(f"[*] Available ONNX Providers: {available_providers}")

providers = []
if 'TensorrtExecutionProvider' in available_providers:
    providers.append(('TensorrtExecutionProvider', trt_options))
if 'CUDAExecutionProvider' in available_providers:
    providers.append('CUDAExecutionProvider')
providers.append('CPUExecutionProvider')

# 3. Create InferenceSession with ONNX model & TensorRT Provider
try:
    session = ort.InferenceSession(onnx_path, providers=providers)
    print(f"[+] Successfully loaded ONNX Session with Providers: {session.get_providers()}")
except Exception as e:
    print(f"[*] Primary provider configuration notice ({e}). Falling back to CPUExecutionProvider...")
    session = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    print(f"[+] Fallback Session loaded: {session.get_providers()}")

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name


### 3. Initialize JetRacer Hardware (`NvidiaRacecar`)


In [ ]:
car = NvidiaRacecar()
car.steering = 0.0
car.throttle = 0.0
print("[+] JetRacer Hardware initialized.")


### 4. Camera-Only Reverse & Evade Control Loop (Zero Distance Sensor)


In [ ]:
if 'ros_sub' in globals() and ros_sub is not None:
    try:
        ros_sub.unregister()
    except Exception:
        pass

state_widget = widgets.ToggleButtons(options=['STOP', 'RUN'], description='Drive State', value='STOP')
camera_html_widget = widgets.HTML(
    value="<p><b>Waiting for ROS Camera Topic...</b></p>",
    layout=widgets.Layout(width='240px', height='240px')
)

latest_ros_image = None
_lock = threading.Lock()

# Bulletproof Rotational Escape Variables
threshold = 0.5
CONFIRM_FRAMES = 3          # Must detect BLOCKED for 3 consecutive frames
blocked_frame_count = 0     # Frame debounce counter
maneuver_state = 'DRIVE'    # States: 'DRIVE', 'REVERSE_TURNING', 'PAUSE', 'CHECK_FORWARD'
maneuver_start_time = 0.0

def ros_image_to_cv2(msg):
    im = np.frombuffer(msg.data, dtype=np.uint8).reshape(msg.height, msg.width, -1)
    if msg.encoding in ['rgb8', 'rgb8']:
        im = cv2.cvtColor(im, cv2.COLOR_RGB2BGR)
    elif msg.encoding == 'rgba8':
        im = cv2.cvtColor(im, cv2.COLOR_RGBA2BGR)
    elif msg.encoding == 'bgra8':
        im = cv2.cvtColor(im, cv2.COLOR_BGRA2BGR)
    if im.shape[0] != 224 or im.shape[1] != 224:
        im = cv2.resize(im, (224, 224))
    return im

def camera_callback(msg):
    global blocked_frame_count, maneuver_state, maneuver_start_time

    if state_widget.value != 'RUN':
        car.throttle = 0.0
        car.steering = 0.0
        maneuver_state = 'DRIVE'
        blocked_frame_count = 0
        return

    if not _lock.acquire(blocking=False):
        return

    try:
        cv_image = ros_image_to_cv2(msg)
        
        # 1. Preprocess & Pure ONNX Runtime TensorRT Inference
        img_input = preprocess_onnx(cv_image)
        outputs = session.run([output_name], {input_name: img_input})
        probs = outputs[0]  # shape (1, 2)
        
        # Compute Softmax probabilities with NumPy
        exp_probs = np.exp(probs - np.max(probs, axis=1, keepdims=True))
        softmax_probs = exp_probs / np.sum(exp_probs, axis=1, keepdims=True)

        # Class 0: 'blocked', Class 1: 'free'
        prob_blocked = float(softmax_probs[0][0])
        prob_free    = float(softmax_probs[0][1])

        now = time.time()

        # 2. Debounce: Confirm BLOCKED only after 3 consecutive frames
        if prob_blocked >= threshold:
            blocked_frame_count += 1
        else:
            blocked_frame_count = max(0, blocked_frame_count - 1)

        # 3. Bulletproof Rotational Escape State Machine
        if maneuver_state == 'DRIVE':
            if blocked_frame_count >= CONFIRM_FRAMES:
                # Confirmed BLOCKED -> Trigger Rotational Reverse (90-degree sharp reverse turn)
                maneuver_state = 'REVERSE_TURNING'
                maneuver_start_time = now
                car.steering = -1.0   # Hard steer left
                car.throttle = -0.22  # Strong reverse throttle
            else:
                # Path FREE -> Drive Straight
                car.steering = 0.0
                car.throttle = 0.18

        elif maneuver_state == 'REVERSE_TURNING':
            # Reverse for 1.8 seconds with max steer angle to rotate car nose 90 degrees
            if now - maneuver_start_time < 1.8:
                car.steering = -1.0
                car.throttle = -0.22
            else:
                # Transition to PAUSE
                maneuver_state = 'PAUSE'
                maneuver_start_time = now
                car.steering = 0.0
                car.throttle = 0.0

        elif maneuver_state == 'PAUSE':
            # Hardware pause for 0.3s for smooth direction change
            if now - maneuver_start_time < 0.3:
                car.steering = 0.0
                car.throttle = 0.0
            else:
                # Transition to CHECK_FORWARD
                maneuver_state = 'CHECK_FORWARD'
                maneuver_start_time = now
                car.steering = 0.0
                car.throttle = 0.18

        elif maneuver_state == 'CHECK_FORWARD':
            # Drive straight forward in new direction for 1.2 seconds
            if now - maneuver_start_time < 1.2:
                car.steering = 0.0
                car.throttle = 0.18
            else:
                # Check if new path is clear
                if blocked_frame_count < CONFIRM_FRAMES:
                    maneuver_state = 'DRIVE'
                    blocked_frame_count = 0
                else:
                    # Still blocked in corner -> Repeat REVERSE_TURNING for another 90 degrees!
                    maneuver_state = 'REVERSE_TURNING'
                    maneuver_start_time = now
                    car.steering = -1.0
                    car.throttle = -0.22

        # 4. Live HTML UI Update
        b64 = base64.b64encode(bgr8_to_jpeg(cv_image)).decode('utf-8')
        status_color = "#ff0000" if blocked_frame_count >= CONFIRM_FRAMES or maneuver_state != 'DRIVE' else "#00ff00"
        status_text  = f"BLOCKED ({blocked_frame_count}/{CONFIRM_FRAMES} frames)" if prob_blocked >= threshold else "FREE"
        html_str = f'''
        <div style="font-family: monospace; background: #1e1e1e; padding: 8px; border-radius: 6px; display: inline-block;">
            <h5 style="margin:0 0 4px 0; color: #ffffff;">Rotational Escape (3-Frame Confirm + 90-Deg Turn)</h5>
            <p style="margin:2px 0; color:{status_color}; font-size:12px;"><b>State:</b> {status_text} | <b>Maneuver:</b> {maneuver_state}</p>
            <p style="margin:2px 0; color:#00ff00; font-size:11px;"><b>Prob Blocked:</b> {prob_blocked:.3f} | <b>Confirm:</b> {blocked_frame_count}/{CONFIRM_FRAMES}</p>
            <p style="margin:2px 0; color:#00ff00; font-size:11px;"><b>Steer:</b> {car.steering:+.2f} | <b>Throttle:</b> {car.throttle:+.2f}</p>
            <img src="data:image/jpeg;base64,{b64}" style="width:224px; height:224px; border:2px solid {status_color}; border-radius:4px; margin-top:4px;" />
        </div>
        '''
        camera_html_widget.value = html_str
    except Exception as e:
        pass
    finally:
        try:
            _lock.release()
        except RuntimeError:
            pass

topic_name = "/csi_cam_0/image_raw"
ros_sub = rospy.Subscriber(topic_name, ROSImage, camera_callback, queue_size=1, buff_size=2**24)

ui_layout = widgets.VBox([
    state_widget,
    camera_html_widget
])

display(ui_layout)
print(f"[*] Subscribed to ROS Camera Topic: {topic_name}")


### 5. Emergency Stop Cell


In [ ]:
state_widget.value = 'STOP'
if 'ros_sub' in globals() and ros_sub is not None:
    try:
        ros_sub.unregister()
    except Exception:
        pass
car.steering = 0.0
car.throttle = 0.0
print("[*] Robot Emergency Stopped.")
